# 01 — What is RAG

**RAG** (Retrieval-Augmented Generation) is: search your own files first, then ask the model to answer from those passages. You do not train the model on the wiki. You attach the right text at question time.

```
question
    → retrieve (find relevant passages)
    → augment (put them in the prompt)
    → generate (answer only from that evidence)
```

| Word | Meaning |
| --- | --- |
| Retrieve | Find the paragraphs that match the question |
| Augment | Add those paragraphs to the prompt |
| Generate | Write the answer from that evidence, or abstain |

An LLM only knows its training data. Northline's labels are not in that memory. That is why RAG exists.

**Scenario.** A banker asks: *What labels does Northline use for information classification?*

1. Read the private policy file.
2. Ask the LLM with no documents attached.
3. See that it cannot know Northline's five labels.

Any OpenAI-compatible model works (`LLM_API_KEY` + `LLM_BASE_URL` + `LLM_MODEL`). This tutorial uses **xAI** via `XAI_API_KEY`.

Context:

- RAG is not fine-tuning. The model weights stay the same.
- RAG is not a database. It is a search-and-answer pattern over evidence.
- RAG quality depends on retrieval, chunking, access control, and evaluation.

Edge cases:

- If the corpus is missing the answer, the assistant should abstain.
- If the retrieved passage is stale, the assistant can confidently answer wrong.
- If a user asks for secrets or PII, retrieval must not become a lookup API.


Definitions and context:

| Concept | Meaning |
| --- | --- |
| Parametric memory | Knowledge stored in model weights during training |
| Context window | Text you send with the current request |
| Corpus | The private files you want the assistant to use |
| Retrieval | Search that selects the most relevant chunks |
| Augmentation | Adding retrieved chunks to the prompt |
| Abstention | Refusing to answer when evidence or authorization is missing |

RAG is valuable when the answer changes often, is private, or needs citations.
It is weaker when the answer requires calculations, transactions, or guaranteed
freshness from a system of record. In those cases RAG should call a tool or
link to the source system instead of pretending the index is authoritative.


In [1]:
import _course
from _course import ROOT, llm, skip_if_no_key


course: Northline secure RAG tutorial
llm   : False | model: grok-4.6


In [2]:
QUESTION = "What labels does Northline Trust use for information classification?"
policy = (ROOT / "data" / "policies" / "information-classification.md").read_text()
print(policy)


# Information Classification Standard

Northline Trust Company labels every record before it is stored, shared, or
retrieved by an assistant. Classification is need-to-know, not job title alone.

## Labels

| Label | Meaning | Who may retrieve it |
| ----- | ------- | ------------------- |
| Public | Marketing, published rates | Anyone |
| Internal | How we work, not client-identifying | Employees |
| Confidential | Client, account, or control data | Advisor, compliance, security, executive |
| Restricted | Privileged access, crypto material | Security, executive |
| MNPI | Material nonpublic information | Compliance, executive only |

A lower-cleared role must not receive a higher-labeled chunk, even if semantic
search ranks it first. Retrieval enforces this; the model is not trusted to
keep secrets.

## Handling rules

- Do not paste Confidential or above into unsanctioned chat tools.
- Assistants that answer staff questions must log role, doc_id, and chunk_id
  for every retrieval.


In [3]:
if skip_if_no_key("ungrounded question"):
    print("Without the file, expect a hedge or a generic ISO-style list — not Northline's five labels.")
else:
    print(llm(QUESTION, system="Answer. If you do not know this bank's policy, say so."))


No LLM key — skipping ungrounded question. Set XAI_API_KEY (this tutorial) or LLM_API_KEY.
Without the file, expect a hedge or a generic ISO-style list — not Northline's five labels.


Next: `02-grounding.ipynb` — put the policy in the prompt and ask again.
